# AUSA attorney tracker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/ausa-attorney-tracker/blob/main/ausa_attorney_tracker.ipynb)

**The question:** did the D.C. U.S. Attorney's Office lose a bigger share of its career attorneys than the rest of the country did?

**The answer: kinda — and the reason it's only "kinda" is the interesting part.** The public data has a "DC" bucket, and that bucket is down about the same amount as everywhere else (~88% vs. ~86% of its Nov 2024 headcount). But "DC" is not the D.C. U.S. Attorney's Office. It's two different organizations added together, and the field that would separate them is redacted. This notebook shows exactly how far you can push the question before the data stops answering — and quantifies the gap it leaves.

Monthly career-attorney headcount for Assistant U.S. Attorneys (DOJ, Executive Office for U.S. Attorneys and the Offices of the U.S. Attorneys, occupational series 0905), queried **live** from the public OPM/EHRI mirror on HuggingFace ([`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data)) with DuckDB over HTTPS. Nothing is downloaded to disk.

**Scope: DC vs. rest-of-country, not state-by-state.** Every geographic field in this data (`duty_station_state_abbreviation`, `duty_station_city`, `core_based_statistical_area`) is privacy-redacted for ~91% of AUSA records — a suppression rule applied uniformly across the whole location hierarchy. DC is the one exception. So the only two honest "area" buckets available are **DC** and **rest-of-country (aggregate)** — this notebook does not fabricate state-level detail the data doesn't actually contain.

**And "DC" is two organizations, not one.** The DC bucket is everyone in this part of DOJ whose duty station is Washington, and two things fit that description: the U.S. Attorney's Office for D.C. (one of the 93 district offices) and EOUSA, the national headquarters that oversees all 93 — also in Washington. `agency_subelement` gives them one label. Part of headquarters is identifiable and is excluded from every table below (see the config cell). The rest isn't: after removing everyone we can identify as HQ, DC still holds ~160 more career attorneys than DOJ says the entire D.C. office has, and nothing in the data separates those ~160 from line prosecutors. The [last section](#can-you-say-anything-about-the-dc-us-attorneys-office) works out what that does and doesn't let you conclude.

In [ ]:
# Setup — installs duckdb/pandas/great_tables if missing (all pip-installable on Colab)
for pkg, mod in [("duckdb", "duckdb"), ("pandas", "pandas"), ("great_tables", "great_tables")]:
    try:
        __import__(mod)
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import duckdb
import pandas as pd
from great_tables import GT, style, loc
import urllib.request
import json
import re

In [ ]:
# Config
REPO = "impactproject/opm-ehri-data"
HF = f"https://huggingface.co/datasets/{REPO}/resolve/main/"
AGENCY_SUBELEMENT = "EXECUTIVE OFFICE FOR U.S. ATTORNEYS AND THE OFFICES OF THE U.S. ATTORNEYS"
SERIES_CODE = "0905"  # attorney

# Nov 2024 (pre-inauguration baseline) through present. Employment snapshots
# are the full federal workforce each month (26-75 MiB per file) filtered
# down to ~6,000 AUSA rows -- within this narrow ~20-month window, pulling
# every single one is still fast (~20s) and doesn't trip HuggingFace's rate
# limit (that only happened pulling the FULL 2015-present history, ~140
# employment files, in an earlier version of this notebook).
# EMPLOYMENT_MONTH_STRIDE is a fixed, deterministic interval (not a
# statistical sample) -- raise it above 1 if you widen START_YM/END_YM back
# toward full history and want to keep the query fast.
START_YM = "202411"
END_YM = "202612"
EMPLOYMENT_MONTH_STRIDE = 1  # 1 = every available month

# appointment_type values that mean "political appointee," excluded below so
# the table tracks the career AUSA workforce. Confirmed by enumerating every
# distinct appointment_type actually present for this population Nov 2024-
# present (2026-08-08) and checking each one, not guessed:
#   - SCHEDULE C: the standard political-appointee schedule (5 CFR 213.3301)
#   - NONCAREER (SENIOR EXECUTIVE SERVICE PERMANENT): noncareer SES = political
#   - EXECUTIVE (EXCEPTED SERVICE NONPERMANENT): always pay_plan_code='AD',
#     grade='40', supervisory_status='SUPERVISOR OR MANAGER' -- one specific,
#     consistent combination, consistent with the (Presidentially-appointed,
#     Senate-confirmed) U.S. Attorney / top leadership slot per district, not
#     an ordinary career attorney classification.
# NOTE: "OTHER (EXCEPTED SERVICE NONPERMANENT)" is NOT excluded even though
# "NONPERMANENT" sounds temporary -- checked against the accessions data and
# it's 90-99.7% of AUSA hires every year 2015-2024 (79.7% in 2025), i.e. the
# ordinary hiring code, not a marker of temporary/surge staffing. It does
# drop to 7.1% of hires in 2026, so the hiring authority for new AUSAs
# evidently changed this year -- noted because it breaks the rule of thumb,
# but it doesn't affect the headcount tables, which count everyone except
# the three political categories above.
POLITICAL_APPOINTMENT_TYPES = [
    "SCHEDULE C (EXCEPTED SERVICE NONPERMANENT)",
    "NONCAREER (SENIOR EXECUTIVE SERVICE PERMANENT)",
    "EXECUTIVE (EXCEPTED SERVICE NONPERMANENT)",
]

# --- Separating EOUSA headquarters from the D.C. U.S. Attorney's Office ---
#
# `agency_subelement` lumps EOUSA's national HQ (in DC) together with all 93
# U.S. Attorney's offices, so the raw "DC" bucket is a mix of HQ staff and
# actual D.C. line prosecutors. Personnel office 4261 is the one slice of HQ
# that IS identifiable, and it's broken out separately in every table below.
# Three signals say it's HQ rather than a field office, and they are NOT of
# equal strength:
#   - pay plan (the strong one): 100% GS/ES/SL, where every other DC career
#     attorney is 100% AD, DOJ's attorney schedule. Exceptionless across all
#     20 months -- 803 person-months here, zero AD; 8,833 in the rest of DC,
#     zero non-AD. Two pay systems that never mix are two organizations.
#   - supervisory share: 40.9% supervisor or manager vs. 12.4% for the rest
#     of DC (Nov 2024) -- top-heavy the way a headquarters is
#   - geography (the weak one): 44 of the 61 people nationwide with this code
#     are in DC, the other 17 scattered 1-3 at a time across 13 states. A few
#     sit in towns that plainly aren't district seats (Old Saybrook CT,
#     Saratoga Springs NY, Collegeville PA), but most of the rest are in
#     cities that ARE district seats or major USAO locations (Columbia SC,
#     Houston, Chicago, Minneapolis). Mildly corroborating, not proof.
KNOWN_HQ_POI = "4261"

# DOJ's own published count for USAO-DC, used in the final section to size
# what's left over after removing personnel office 4261. Verbatim from
# https://www.justice.gov/usao-dc/about-us (checked 2026-08-09): "It is the
# largest United States Attorney's Office with over 330 Assistant United
# States Attorneys and over 330 support personnel." Treat this as a rough,
# undated, self-reported floor -- boilerplate on an About page, not a
# point-in-time figure -- which is exactly why the last section uses it to
# bound the answer rather than to compute one.
DOJ_PUBLISHED_USAO_DC_AUSAS = 330

In [ ]:
def list_all_files(repo=REPO):
    """Every file in the HF tree, following pagination.

    The tree API caps a single response at 1000 entries (Link header,
    rel="next", cursor-based). This repo already has 1000+ files across
    accessions/employment/separations combined — a one-shot fetch silently
    truncates whichever directory sorts last alphabetically (separations/).
    """
    url = f"https://huggingface.co/api/datasets/{repo}/tree/main?recursive=true&limit=1000"
    out = []
    while url:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req) as r:
            headers = dict(r.getheaders())
            out.extend(json.load(r))
        link = headers.get("Link")
        m = re.search(r'<([^>]+)>;\s*rel="next"', link) if link else None
        url = m.group(1) if m else None
    return out


def monthly_urls(files, dataset, start, end):
    """Latest version per month, from `start` through `end` (YYYYMM strings)."""
    best = {}
    for f in files:
        m = re.search(dataset + r"_(\d{6})_v(\d+)\.parquet", f["path"])
        if not m:
            continue
        month, ver = m.group(1), int(m.group(2))
        if start <= month <= end and (month not in best or ver > best[month][0]):
            best[month] = (ver, f["path"])
    return [HF + best[m][1] for m in sorted(best)]


files = list_all_files()
print(f"{len(files)} files total in the HF repo")

In [ ]:
con = duckdb.connect()
# DuckDB errors out on this if it thinks it's in Jupyter but ipywidgets isn't
# installed -- it can't render the widget-based progress bar and refuses to
# change the setting at all. Colab ships ipywidgets so this never surfaces
# there; a plain local kernel without it would otherwise fail on line 2 of
# the notebook's main query cell. Suppressing progress bars is cosmetic, so
# failing to suppress them is not worth stopping for.
try:
    con.execute("SET enable_progress_bar=false;")
except duckdb.Error:
    pass
con.execute("INSTALL httpfs; LOAD httpfs;")
# Safety net, not the primary defense — the real fix against HuggingFace's
# rate limit is querying far fewer/smaller files in the first place (see
# START_YM/EMPLOYMENT_MONTH_STRIDE above). If this cell still errors with
# HTTP 429 after all retries, just re-run it — it's a temporary throttle.
con.execute("SET http_retries=6;")
con.execute("SET http_retry_wait_ms=1000;")
con.execute("SET http_retry_backoff=2;")
con.execute("SET threads=4;")

# Three segments, not two. Splitting the identifiable EOUSA HQ people
# (KNOWN_HQ_POI) out of DC rather than leaving them inside it is the one
# HQ-vs-field correction the data actually supports; the rest of the HQ
# population can't be told apart from line prosecutors at all (final
# section). Names are spelled out here because they become column headers.
DC_FIELD = "DC (excl. identified HQ)"
DC_HQ = "DC identified EOUSA HQ"
REST = "Rest of country"
SEGMENT_CASE = f"""CASE
    WHEN duty_station_state_abbreviation <> 'DC' THEN '{REST}'
    WHEN personnel_office_identifier_code = '{KNOWN_HQ_POI}' THEN '{DC_HQ}'
    ELSE '{DC_FIELD}'
END"""
POLITICAL_EXCLUSION_SQL = "(" + ",".join(f"'{t}'" for t in POLITICAL_APPOINTMENT_TYPES) + ")"


def query_employment(stride=EMPLOYMENT_MONTH_STRIDE):
    """Monthly (ym, area, headcount) totals for the career AUSA population.

    `stride` pulls every Nth available employment file instead of every one
    -- a fixed, deterministic interval, not a statistical sample. Filters
    on `snapshot_yyyymm BETWEEN START_YM AND END_YM` explicitly, not just
    on which files get fetched, and excludes POLITICAL_APPOINTMENT_TYPES
    (see config cell) so headcount figures reflect the career AUSA
    workforce, not political leadership turnover.
    """
    urls = monthly_urls(files, "employment", START_YM, END_YM)[::stride]
    lst = "[" + ",".join(f"'{u}'" for u in urls) + "]"
    return con.execute(f"""
        SELECT snapshot_yyyymm AS ym,
               {SEGMENT_CASE} AS area,
               SUM(TRY_CAST(count AS BIGINT)) AS headcount
        FROM read_parquet({lst}, union_by_name=true)
        WHERE agency_subelement = '{AGENCY_SUBELEMENT}'
          AND occupational_series_code = '{SERIES_CODE}'
          AND snapshot_yyyymm BETWEEN '{START_YM}' AND '{END_YM}'
          AND appointment_type NOT IN {POLITICAL_EXCLUSION_SQL}
        GROUP BY 1, 2
    """).df()

## Query employment

Pulls every available month Nov 2024–present by default
(`EMPLOYMENT_MONTH_STRIDE = 1`). Employment snapshots are the full federal
workforce each month (26–75 MiB each) filtered down to ~6,000 AUSA rows --
raise the stride if you widen the date range back toward full history and
want to keep the query fast.

In [ ]:
employment_interval_desc = (
    "every available month" if EMPLOYMENT_MONTH_STRIDE == 1
    else f"every {EMPLOYMENT_MONTH_STRIDE} months"
)
print(f"Querying employment (headcount), {employment_interval_desc}...")
df_employment = query_employment()
print(f"  {len(df_employment)} (month, area) rows, {df_employment.ym.nunique()} distinct months")

## Table

[great_tables](https://posit-dev.github.io/great-tables/) instead of a
matplotlib heatmap — a color-shaded table reads more clearly than an
`imshow` grid at this size, with real numbers in every cell instead of tiny
rotated-axis labels. One row per month, DC and rest-of-country as separate
columns, each also shown indexed to the Nov 2024 baseline (=100%) so loss
reads as a percentage. Diverging red(loss)/blue(gain) color on the %
columns only — see `build_employment_gt`'s docstring for why the raw
counts are left uncolored.

The DC column **excludes** the identifiable EOUSA headquarters personnel
office (`KNOWN_HQ_POI`, ~44 people falling to ~37) — the one HQ-vs-field
correction this data supports. It barely moves the line, which is the
point: the piece of the problem that could be fixed wasn't the piece that
mattered. See the [last section](#can-you-say-anything-about-the-dc-us-attorneys-office).

In [ ]:
DIVERGING_PALETTE = ["#B2182B", "#F7F7F7", "#2166AC"]  # red (below baseline) - white - blue (above baseline)
ATTY_NOTE = "career attorneys only, series 0905 — excludes political appointees"
SOURCE_NOTE = (
    "Source: OPM/EHRI (impactproject/opm-ehri-data on HuggingFace), queried live. "
    "DC vs. rest-of-country only — finer geography (state/city) is privacy-redacted "
    "for this population. \"DC\" still mixes EOUSA headquarters with the D.C. U.S. "
    "Attorney's Office beyond the HQ personnel office excluded here."
)

# Fixed, generous domain for the % columns below -- NOT derived from this
# window's own min/max. A domain scaled to just this window's own extremes
# makes the worst month always look maximally saturated no matter how mild
# the real swing is (a 9% decline looked nearly as dark red as a 14% one
# when the domain was [85,100]). Anchoring to a fixed, meaningfully-extreme
# reference instead means color intensity reflects how bad things actually
# are, not just "worst cell inside whatever window I happen to be looking
# at right now."
HEADCOUNT_PCT_DOMAIN = [75, 125]  # full color at a +/-25pp swing from baseline headcount


def to_area_table(df, value_col, areas=(DC_FIELD, REST)):
    """One row per month, one column per area in `areas`.

    Defaults to the two comparison groups only -- DC_HQ is deliberately not
    a third column here (it's ~40 people, it isn't part of the question the
    table is answering, and it gets its own treatment in the final
    section). No Total column either: rest-of-country outnumbers DC ~10:1,
    so Total just mirrors rest-of-country's pattern almost exactly and adds
    a third column of numbers without adding a third story. Month is
    formatted "Mon YYYY" (e.g. "Nov 2024") -- the raw "202411" YYYYMM
    string is hard to read.
    """
    pivot = df.pivot_table(index="ym", columns="area", values=value_col, aggfunc="sum")
    for c in areas:
        if c not in pivot.columns:
            pivot[c] = pd.NA
    tbl = pivot[list(areas)].sort_index().reset_index().rename(columns={"ym": "Month"})
    tbl["Month"] = pd.to_datetime(tbl["Month"], format="%Y%m").dt.strftime("%b %Y")
    return tbl


def build_employment_gt(df, cadence_note=None):
    """Headcount table. Raw counts are left UNCOLORED on purpose --
    coloring them on their own scale while the % columns use a diverging
    scale told two contradictory stories with the same dark color (dark =
    high count = good, vs. dark = far from baseline = bad), right next to
    each other in the same row. Only the % columns carry color, both
    sharing ONE fixed domain (HEADCOUNT_PCT_DOMAIN), so the same percentage
    always renders as the same shade regardless of which area column it's
    in, DC's shade is directly comparable to rest-of-country's, and the
    color intensity means the same thing across different runs.

    Count columns are labeled "Count", not repeating the area name -- the
    spanner above each pair already says that. cols_width keeps the table
    close to its content width instead of the browser/notebook stretching
    it to fill the cell.

    `cadence_note` overrides the default "Every available month" /
    "Every N months" subtitle text -- used by the key-months table below,
    which passes a df already filtered down to a handful of rows.
    """
    tbl = to_area_table(df, "headcount")
    baseline_month = tbl.iloc[0]["Month"]
    count_cols = [DC_FIELD, REST]
    idx_cols = [f"{c}_idx" for c in count_cols]
    for col in count_cols:
        tbl[f"{col}_idx"] = tbl[col] / tbl.iloc[0][col] * 100

    if cadence_note is None:
        cadence_note = (
            "Every available month" if EMPLOYMENT_MONTH_STRIDE == 1
            else f"Every {EMPLOYMENT_MONTH_STRIDE} months"
        )
    gt = (
        GT(tbl)
        .tab_header(
            title=f"AUSA headcount ({ATTY_NOTE})",
            subtitle=f"{cadence_note}. % columns indexed to the {baseline_month} baseline (=100%), shared color scale",
        )
        .tab_source_note(SOURCE_NOTE)
        .fmt_integer(columns=count_cols)
        .fmt_number(columns=idx_cols, decimals=0, pattern="{x}%")
        .cols_label(**{c: "Count" for c in count_cols}, **{f"{c}_idx": "% of baseline" for c in count_cols})
        .cols_width({
            "Month": "90px",
            DC_FIELD: "70px", f"{DC_FIELD}_idx": "100px",
            REST: "110px", f"{REST}_idx": "100px",
        })
        .data_color(columns=idx_cols, palette=DIVERGING_PALETTE, domain=HEADCOUNT_PCT_DOMAIN)
    )
    for col in count_cols:
        gt = gt.tab_spanner(label=col, columns=[col, f"{col}_idx"])
    return gt

In [ ]:
build_employment_gt(df_employment)

## Key-months summary

A compact 4-row version of the same table above — useful for a screenshot
where the full monthly table is too tall (e.g. a social post). Picks the
baseline month, two fixed narratively-relevant months, and whatever the
latest available month is (so this stays current as new data arrives
without editing `KEY_MONTHS` by hand).

In [ ]:
# 202502 = DC's sharp early drop (99% -> 92% while rest-of-country was
# still at 98%); 202509 = the shared national low point that rest-of-country
# then caught down to. START_YM and the latest available month anchor the
# two ends of the story and update automatically.
KEY_MONTHS = [START_YM, "202502", "202509", df_employment.ym.max()]
df_key_months = df_employment[df_employment.ym.isin(KEY_MONTHS)]
build_employment_gt(df_key_months, cadence_note="4 key months")

## Can you say anything about the D.C. U.S. Attorney's Office?

Everything above is about a bucket called "DC." The question people actually
want answered is about the **D.C. U.S. Attorney's Office**, and those are not
the same thing. The DC bucket is everyone in this corner of DOJ whose duty
station is Washington — and two different organizations fit that description:
the U.S. Attorney's Office for D.C., an ordinary district office that happens
to be in the capital, and EOUSA, the national headquarters that oversees all
93 district offices, also in Washington. `agency_subelement` gives them a
single label.

Part of headquarters is identifiable and has already been removed from the
tables above: personnel office `4261`, on 100% GS/ES/SL pay plans where every
other DC career attorney is on DOJ's AD attorney schedule — a split with no
exceptions in any of the 20 months — and 40.9% supervisors versus 12.4% for
the rest of DC.

The table below walks the arithmetic down from the raw DC bucket and shows
that removing it doesn't close the gap: what's left is still far more
attorneys than DOJ says the entire D.C. office has.

In [ ]:
def month_label(ym):
    return pd.to_datetime(ym, format="%Y%m").strftime("%b %Y")


def segment_counts(df, ym):
    """{segment name: headcount} for one month."""
    s = df[df.ym == ym].set_index("area")["headcount"]
    return {k: float(s.get(k, 0)) for k in (DC_FIELD, DC_HQ, REST)}


BASELINE_YM, LATEST_YM = df_employment.ym.min(), df_employment.ym.max()
base, latest = segment_counts(df_employment, BASELINE_YM), segment_counts(df_employment, LATEST_YM)

# A reconciliation, deliberately laid out as one subtraction per row rather
# than as a single "gap = X" number -- the whole point is which step closes
# the gap and which doesn't, and that's only visible if each step is its own
# line. Months are columns because there are only two of them and the
# interesting comparison is down a column, not across.
recon = pd.DataFrame({
    "Step": [
        "DC, all career attorneys (series 0905)",
        f"− Identified EOUSA HQ (personnel office {KNOWN_HQ_POI})",
        "= DC, excluding identified HQ",
        "− DOJ's published USAO-DC AUSA count (\"over 330\")",
        "= Unexplained excess",
    ],
    month_label(BASELINE_YM): [
        base[DC_FIELD] + base[DC_HQ], base[DC_HQ], base[DC_FIELD],
        DOJ_PUBLISHED_USAO_DC_AUSAS, base[DC_FIELD] - DOJ_PUBLISHED_USAO_DC_AUSAS,
    ],
    month_label(LATEST_YM): [
        latest[DC_FIELD] + latest[DC_HQ], latest[DC_HQ], latest[DC_FIELD],
        DOJ_PUBLISHED_USAO_DC_AUSAS, latest[DC_FIELD] - DOJ_PUBLISHED_USAO_DC_AUSAS,
    ],
})

(
    GT(recon)
    .tab_header(
        title="\"DC\" is bigger than the D.C. U.S. Attorney's Office, even after removing the HQ we can identify",
        subtitle="Career attorneys, series 0905, political appointees excluded. Counts as of each month.",
    )
    .tab_source_note(
        "Sources: OPM/EHRI (impactproject/opm-ehri-data on HuggingFace), queried live; "
        "USAO-DC figure from justice.gov/usao-dc/about-us (\"over 330 Assistant United "
        "States Attorneys\"), an undated self-reported number treated here as a rough floor."
    )
    .fmt_integer(columns=[month_label(BASELINE_YM), month_label(LATEST_YM)])
    .cols_width({"Step": "340px", month_label(BASELINE_YM): "100px", month_label(LATEST_YM): "100px"})
    .tab_style(
        style=style.text(weight="bold"),
        locations=loc.body(rows=[4]),
    )
)

### So we think "DC" is two groups

That leftover — ~160 attorneys in Nov 2024 — is the whole problem. It's too
big to be noise and too big to be explained by DOJ's figure being a little
stale. The most likely explanation is that EOUSA headquarters employs a
substantial number of attorneys on the same AD attorney pay schedule, in the
same city, under the same `agency_subelement`, whose personnel office code is
redacted along with everyone else's.

The reason they can't be pulled out is that the leftover group is
*homogeneous* on everything this data exposes. Unlike personnel office
`4261`, which announced itself with a different pay system, these 491 people
are 100% AD pay plan, 100% duty city Washington, ~99% excepted service, and
split across the two ordinary AUSA appointment codes. There's no seam to cut
along.

So the DC column is a sum of two populations, in unknown proportion, moving
at unknown relative rates. **What that costs you is the ability to answer the
original question.** The table below takes the observed decline in the DC
column and allocates it three different ways between the two hidden groups —
all of them arithmetically consistent with the published data:

- **all of it hit the office**, and headquarters was flat
- **headquarters shrank at the rate we can actually see** it shrink in the
  identifiable HQ personnel office, and the office absorbed the remainder
- **all of it hit headquarters**, and the office was flat

The spread between the first and third is the width of what the data doesn't
tell you.

In [ ]:
# Scenario months: the point where DC looked like it was diverging most from
# the rest of the country (Feb 2025), and the latest month. If the early-drop
# claim survives the bounding exercise anywhere, it's at 202502.
SCENARIO_MONTHS = ["202502", LATEST_YM]

hidden_hq_baseline = base[DC_FIELD] - DOJ_PUBLISHED_USAO_DC_AUSAS


def usao_dc_scenarios(ym):
    """Possible USAO-DC index values for one month, under three allocations.

    The DC column is office + hidden HQ, and only the sum is observed, so
    the office's own trajectory is not identified -- it's a free parameter
    pinned only by "neither subgroup can lose more than the whole decline."
    These three points are not equally likely; they're the two edges and one
    substantive middle, which is the honest way to report an unidentified
    quantity.
    """
    cur = segment_counts(df_employment, ym)
    decline = base[DC_FIELD] - cur[DC_FIELD]
    office_0 = DOJ_PUBLISHED_USAO_DC_AUSAS

    # Edge 1: headquarters flat, office absorbs everything.
    all_office = office_0 - decline
    # Middle: the hidden HQ group shrinks at the same rate as the HQ group we
    # CAN see (personnel office 4261). Not a law of nature -- an assumption
    # that the two halves of one headquarters behave alike -- but the only
    # scenario here anchored to an observed number rather than to an edge.
    hidden_now = hidden_hq_baseline * (cur[DC_HQ] / base[DC_HQ])
    hq_like_visible = cur[DC_FIELD] - hidden_now
    # Edge 2: office flat, headquarters absorbs everything. Only coherent
    # while the decline is smaller than the hidden group -- otherwise the
    # hidden group would have to go negative.
    all_hq = office_0 if decline <= hidden_hq_baseline else float("nan")

    return {
        "Rest of country (observed)": cur[REST] / base[REST] * 100,
        "DC excl. identified HQ (observed)": cur[DC_FIELD] / base[DC_FIELD] * 100,
        "USAO-DC if the whole DC decline was the office": all_office / office_0 * 100,
        "USAO-DC if hidden HQ shrank like the HQ we can see": hq_like_visible / office_0 * 100,
        "USAO-DC if the whole DC decline was headquarters": all_hq / office_0 * 100,
    }


scen = pd.DataFrame({month_label(ym): usao_dc_scenarios(ym) for ym in SCENARIO_MONTHS})
scen = scen.reset_index().rename(columns={"index": ""})
scen_cols = [month_label(ym) for ym in SCENARIO_MONTHS]

(
    GT(scen)
    .tab_header(
        title="What the D.C. U.S. Attorney's Office could have done, consistent with the same public data",
        subtitle=f"% of {month_label(BASELINE_YM)} baseline. Rows 3-5 are three allocations of one observed decline.",
    )
    .tab_source_note(
        "The DC column is the office plus an unknown number of headquarters attorneys; only the sum "
        "is published. At both dates the rest-of-country figure falls inside the range of possible "
        "USAO-DC values — so this data cannot establish that D.C. was hit harder, or less hard, than "
        "the rest of the country."
    )
    .fmt_number(columns=scen_cols, decimals=0, pattern="{x}%")
    .cols_width({"": "330px", **{c: "110px" for c in scen_cols}})
    .data_color(columns=scen_cols, palette=DIVERGING_PALETTE, domain=HEADCOUNT_PCT_DOMAIN)
    .tab_style(style=style.text(weight="bold"), locations=loc.body(rows=[0, 1]))
)

### Which is why the answer is "kinda"

Read the table down each column. In both months, the rest-of-country number
sits **inside** the range of USAO-DC values the public data allows. The
office could have been hit meaningfully harder than the rest of the country,
or not at all — the same published numbers are consistent with both.

One caveat that cuts in a useful direction. DOJ says "over 330," which is a
floor, not a measurement — strictly, 491 is *consistent* with "over 330." So
the reconciliation's ~160 is best read as the **largest** the hidden
headquarters group can be while DOJ's own number holds. That's also why the
scenario table is built on 330 exactly: the biggest possible hidden group
produces the widest possible range, so the range above is the most
conservative version of "you can't tell." If the D.C. office is really larger
than 330, the hidden group shrinks, the range narrows, and every scenario
moves toward the office having absorbed the decline itself.

None of which is the same as knowing nothing:

- **The DC bucket is real and it's down**, ~12-13% off its Nov 2024 baseline,
  and that's true whether or not you can split it.
- **The early-2025 drop is real in the aggregate.** By Feb 2025 the DC bucket
  was at 92% while the rest of the country was still at 98% — DC's decline
  came first, and the rest of the country caught down to it over the
  following year rather than the reverse.
- **The one headquarters group we can watch barely moved early** (down ~4%
  by Feb 2025, versus ~8% for the DC bucket as a whole). If the hidden
  headquarters attorneys behaved like the ones we can see, most of that early
  drop was the office itself — which is the middle row of the table, and the
  most defensible single read. It just isn't a measurement.

So: the office probably did take an earlier and sharper hit than the rest of
the country, and by mid-2026 the two had converged. But "probably," resting
on an assumption about a group the data hides, is as far as this goes — and
the reason it's as far as it goes isn't the analysis, it's that
`personnel_office_identifier_code` is redacted for 91% of the people who'd
answer the question.

## Caveats

- **"DC" is not the D.C. U.S. Attorney's Office**, even after excluding personnel office `4261` — see the section above. Every DC figure in this notebook is DC-based EOUSA/USAO career attorneys broadly.
- **The scenario table is a bounding exercise, not an estimate.** It leans on DOJ's undated "over 330" boilerplate as a baseline for the office. If that figure is stale or rounded, the width of the range shifts — but not the conclusion that a range is all you get, which follows from the redaction alone.
- **Area is DC vs. rest-of-country only** — see the scope note at the top. This is a real limit of the public EHRI data (privacy suppression), not a limit of this notebook's queries.
- **Employment defaults to every available month** (`EMPLOYMENT_MONTH_STRIDE = 1`). Within this ~20-month window that's fast (~20s) and doesn't trip HuggingFace's rate limit — that only happened pulling the FULL 2015-present history (~140 files) in an earlier version of this notebook. Raise the stride above 1 if you widen `START_YM`/`END_YM` back toward full history and want to keep the query fast; it's a fixed, deterministic interval (always the same months), not a statistical sample.
- **Everything is queried live** — figures may shift slightly as OPM/EHRI publishes revisions (files are versioned; this notebook always takes the latest version per month).
- **`count` is a string column in the source parquet** and is cast with `TRY_CAST` — any row that fails to cast contributes 0, not an error, so a malformed value would silently under-count rather than crash the notebook.